Splitting labels into two groups fake news and reliable.

In [45]:
import pandas as pd
import ast
from rich.progress import track
from sklearn.feature_extraction.text import CountVectorizer
from sklearn import linear_model
from sklearn.metrics import accuracy_score
import swifter
import numpy as np

In [2]:
all_columns = [
    'id', 'domain', 'type', 'url', 'content', 'scraped_at', 'inserted_at', 
    'updated_at', 'title', 'authors', 'keywords', 'meta_keywords', 
    'meta_description', 'tags', 'summary']
converters_dict = {col: ast.literal_eval for col in all_columns}

training_set = pd.read_csv('../Group_project/training_set.csv', converters = converters_dict)

In [3]:
# creating list of 10000 most common words from vocabulary found in part 1, the first row has been removed due to it being 'na'
full_voc = pd.read_csv('../Group_project/vocabulary_995000.csv')
sub_voc = set((full_voc.iloc[1:10001])['word'].values.tolist())
print(sub_voc)

{'nashvill', 'guerra', 'educ', 'hast', 'reconnaiss', 'brennan', 'innoc', 'dj', 'utter', 'freight', 'shave', 'prison', 'hat', 'paramilitari', 'monro', 'sox', 'cede', 'erect', 'impend', 'attract', 'newcastl', 'grasp', 'incess', 'duck', 'sunday', 'adult', 'zionist', 'sole', 'unconvent', 'incit', 'typo', 'refer', 'cruelti', 'mai', 'scold', 'sheep', 'razib', 'unpaid', 'bloodsh', 'undoubt', 'nichola', 'gaug', 'propagandist', 'gave', 'dwarf', 'bulk', 'manila', 'vernon', 'yemen', 'fitzpatrick', 'mph', 'idl', 'predatori', 'faint', 'uncheck', 'lynn', 'key', 'laser', 'wright', 'hijab', 'declar', 'kim', 'hebdo', 'unifi', 'verizon', 'spare', 'stuart', 'instig', 'shopper', 'warfar', 'gps', 'peak', 'blacklist', 'relentless', 'offshor', 'blockbust', 'dictionari', 'companion', 'barron', 'anni', 'panick', 'enjoy', 'stubborn', 'rochest', 'potassium', 'garment', 'parungo', 'lifelong', 'justin', 'shelter', 'medic', 'comfort', 'brutal', 'wave', 'motor', 'jockey', 'consol', 'quip', 'copi', 'kyle', 'paint', '

In [4]:
types = {i[0] for i in training_set['type'] if len(i) < 2}
print(types)
fake_news = {'fake', 'satir', 'bias', 'conspiraci', 'junksci', 'hate', 'rumor'}
reliable_news = {'clickbait', 'unreli', 'polit', 'reliabl'}

def filter_type(type, fake, rel):
    if type in fake:
        return 'fake'
    elif type in rel:
        return 'reliable'

{'unreli', 'clickbait', 'rumor', 'conspiraci', 'bias', 'satir', 'fake', 'reliabl', 'junksci', 'hate', 'polit', 'unknown'}


In [30]:
x_t = [[word for word in str_list if word in sub_voc] for str_list in track(training_set['content'], description = 'processing')]
x_t_s = [' '.join(word_list) for word_list in track(x_t, description = 'processing')]

y_t = [filter_type(t[0], fake_news, reliable_news) for t in track(training_set['type'], description = 'processing')]

Output()

Output()

Output()

In [50]:
x_train = []
y_t_ = []
for i in range(len(y_t)):
    if y_t[i] in {'fake', 'reliable'}:
        x_train.append(x_t_s[i])
        y_t_.append(y_t[i])

vectorizer = CountVectorizer()

X_train = vectorizer.fit_transform(x_train)
y_train = np.array(y_t_)

In [39]:
logr = linear_model.LogisticRegression(max_iter = 100000)

logr.fit(X_train, y_train)

,"penalty penalty: {'l1', 'l2', 'elasticnet', None}, default='l2'Specify the norm of the penalty:- `None`: no penalty is added;- `'l2'`: add a L2 penalty term and it is the default choice;- `'l1'`: add a L1 penalty term;- `'elasticnet'`: both L1 and L2 penalty terms are added... warning:: Some penalties may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionadded:: 0.19 l1 penalty with SAGA solver (allowing 'multinomial' + L1).. deprecated:: 1.8 `penalty` was deprecated in version 1.8 and will be removed in 1.10. Use `l1_ratio` instead. `l1_ratio=0` for `penalty='l2'`, `l1_ratio=1` for `penalty='l1'` and `l1_ratio` set to any float between 0 and 1 for `'penalty='elasticnet'`.",'deprecated'
,"C C: float, default=1.0Inverse of regularization strength; must be a positive float.Like in support vector machines, smaller values specify strongerregularization. `C=np.inf` results in unpenalized logistic regression.For a visual example on the effect of tuning the `C` parameterwith an L1 penalty, see::ref:`sphx_glr_auto_examples_linear_model_plot_logistic_path.py`.",1.0
,"l1_ratio l1_ratio: float, default=0.0The Elastic-Net mixing parameter, with `0 <= l1_ratio <= 1`. Setting`l1_ratio=1` gives a pure L1-penalty, setting `l1_ratio=0` a pure L2-penalty.Any value between 0 and 1 gives an Elastic-Net penalty of the form`l1_ratio * L1 + (1 - l1_ratio) * L2`... warning:: Certain values of `l1_ratio`, i.e. some penalties, may not work with some solvers. See the parameter `solver` below, to know the compatibility between the penalty and solver... versionchanged:: 1.8 Default value changed from None to 0.0... deprecated:: 1.8 `None` is deprecated and will be removed in version 1.10. Always use `l1_ratio` to specify the penalty type.",0.0
,"dual dual: bool, default=FalseDual (constrained) or primal (regularized, see also:ref:`this equation `) formulation. Dual formulationis only implemented for l2 penalty with liblinear solver. Prefer `dual=False`when n_samples > n_features.",False
,"tol tol: float, default=1e-4Tolerance for stopping criteria.",0.0001
,"fit_intercept fit_intercept: bool, default=TrueSpecifies if a constant (a.k.a. bias or intercept) should beadded to the decision function.",True
,"intercept_scaling intercept_scaling: float, default=1Useful only when the solver `liblinear` is usedand `self.fit_intercept` is set to `True`. In this case, `x` becomes`[x, self.intercept_scaling]`,i.e. a ""synthetic"" feature with constant value equal to`intercept_scaling` is appended to the instance vector.The intercept becomes``intercept_scaling * synthetic_feature_weight``... note:: The synthetic feature weight is subject to L1 or L2 regularization as all other features. To lessen the effect of regularization on synthetic feature weight (and therefore on the intercept) `intercept_scaling` has to be increased.",1
,"class_weight class_weight: dict or 'balanced', default=NoneWeights associated with classes in the form ``{class_label: weight}``.If not given, all classes are supposed to have weight one.The ""balanced"" mode uses the values of y to automatically adjustweights inversely proportional to class frequencies in the input dataas ``n_samples / (n_classes * np.bincount(y))``.Note that these weights will be multiplied with sample_weight (passedthrough the fit method) if sample_weight is specified... versionadded:: 0.17 *class_weight='balanced'*",None
,"random_state random_state: int, RandomState instance, default=NoneUsed when ``solver`` == 'sag', 'saga' or 'liblinear' to shuffle thedata. See :term:`Glossary ` for details.",None
,"solver solver: {'lbfgs', 'liblinear', 'newton-cg', 'newton-cholesky', 'sag', 'saga'}, default='lbfgs'Algorithm to use in the optimization problem. Default is 'lbfgs'.To choose a solver, you might want to consider the following aspects:- 'lbfgs' is a good default solver because it works reasonably well for a wide class of problems.- For :term:`mul

In [40]:
test_set = pd.read_csv('../Group_project/test_set.csv', converters = converters_dict)

In [41]:
xt_t = [[word for word in str_list if word in sub_voc] for str_list in track(test_set['content'], description = 'processing')]
xt_t_s = [' '.join(word_list) for word_list in track(xt_t, description = 'processing')]

yt_t = [filter_type(t[0], fake_news, reliable_news) for t in track(test_set['type'], description = 'processing')]

Output()

Output()

Output()

In [54]:
x_test = []
yt_t_ = []
for i in range(len(yt_t)):
    if yt_t[i] in {'fake', 'reliable'}:
        x_test.append(xt_t_s[i])
        yt_t_.append(yt_t[i])

X_test = vectorizer.transform(x_test)
y_test = np.array(yt_t_)

In [55]:
predictions = logr.predict(X_test)

In [56]:
print(accuracy_score(y_test, predictions))

0.8495512465373961
